# MedVQA — Results & Grad-CAM Analysis
## Model Evaluation and Explainability

This notebook analyzes trained model results, generates calibration plots,
and visualizes Grad-CAM attention heatmaps on medical images.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

# Add project root
sys.path.insert(0, str(Path.cwd().parent))

from src.data.loader import VQARADDataset, collate_fn
from src.data.preprocessor import MedicalImagePreprocessor, TextPreprocessor
from src.evaluation.metrics import compute_all_metrics, compute_ece
from src.evaluation.gradcam import GradCAM
from src.evaluation.evaluator import Evaluator
from src.models.vision_encoder import BioViLTEncoder
from src.models.language_model import MistralQLoRA
from src.models.fusion import CrossAttentionFusion
from src.models.medvqa_model import MedVQAModel
from src.utils.config import MedVQAConfig
from src.inference.pipeline import MedVQAPipeline

print('Libraries loaded')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# Load trained model
config = MedVQAConfig.from_yaml('../configs/default_config.yaml')

# Rebuild model
vision_encoder = BioViLTEncoder(
    model_name=config.model.vision_encoder_name,
    fallback_model_name=config.model.vision_encoder_fallback,
    projection_dim=config.model.projection_dim,
    freeze_encoder=True,
)

language_model = MistralQLoRA(
    model_name=config.model.lm_model_name,
    load_in_4bit=config.model.load_in_4bit,
    lora_r=config.model.lora_r,
    lora_alpha=config.model.lora_alpha,
    gradient_checkpointing=False,
)

fusion = CrossAttentionFusion(
    d_model=config.model.projection_dim,
    n_heads=config.model.fusion_num_heads,
)

medvqa_model = MedVQAModel(
    vision_encoder=vision_encoder,
    language_model=language_model,
    fusion=fusion,
    freeze_vision=True,
    num_beams=config.model.num_beams,
)

# Load checkpoint
checkpoint_path = config.inference.checkpoint_path
if Path(checkpoint_path).exists():
    medvqa_model.load_state_dict(torch.load(checkpoint_path, map_location=device), strict=False)
    print(f'Loaded checkpoint from {checkpoint_path}')
else:
    print(f'No checkpoint found at {checkpoint_path}. Using untrained model.')

In [ ]:
# Evaluate on test set
image_processor = MedicalImagePreprocessor(image_size=config.data.image_size)
text_processor = TextPreprocessor(model_name=config.data.tokenizer_name)

test_dataset = VQARADDataset(
    data_dir=config.data.raw_dir,
    split='test',
    image_transform=image_processor,
    text_preprocessor=text_processor,
)

from torch.utils.data import DataLoader
test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
)

# Evaluate
evaluator = Evaluator(
    model=medvqa_model,
    text_preprocessor=text_processor,
    device=device,
)

results = evaluator.evaluate_split(test_loader, split_name='test', use_mc_dropout=True)
print(f'\nTest metrics: {results["metrics"]}')

In [ ]:
# Calibration Plot
if results['confidences']:
    confidences = np.array(results['confidences'])
    correct = np.array([
        1 if p.strip().lower().rstrip('.!?') == r.strip().lower().rstrip('.!?')
        else 0
        for p, r in zip(results['predictions'], results['references'])
    ])
    
    ece_result = compute_ece(confidences, correct, n_bins=15)
    
    # Plot reliability diagram
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
    
    # Bin stats
    bin_centers = [(b['bin_lower'] + b['bin_upper']) / 2 for b in ece_result['bin_stats']]
    accuracies = [b['accuracy'] for b in ece_result['bin_stats']]
    confs = [b['confidence'] for b in ece_result['bin_stats']]
    
    ax.plot(bin_centers, accuracies, 'o-', color='steelblue', linewidth=2, label='Model')
    ax.fill_between(bin_centers, confs, accuracies, alpha=0.3, color='red', 
                     label='Gap (overconfidence →)')
    
    ax.set_xlabel('Confidence', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(f'Reliability Diagram (ECE = {ece_result["ece"]:.4f})', fontsize=14)
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    
    plt.tight_layout()
    plt.savefig('../experiments/calibration_plot.png', dpi=150)
    plt.show()
    print(f'ECE: {ece_result["ece"]:.4f}')
    print(f'Max calibration error: {ece_result["max_calibration_error"]:.4f}')
else:
    print('No confidence scores available.')

In [ ]:
# Grad-CAM Visualization Examples
gradcam = GradCAM(model=medvqa_model, image_size=config.data.image_size)

fig, axes = plt.subplots(4, 3, figsize=(12, 16))

for idx in range(min(4, len(test_dataset))):
    sample = test_dataset[idx]
    
    # Load original image
    orig_image = Image.open(sample['image_path']).convert('RGB')
    
    # Get preprocessed tensors
    image_tensor = sample['image'].unsqueeze(0).to(device)
    input_ids = sample['input_ids'].unsqueeze(0).to(device)
    attention_mask = sample['attention_mask'].unsqueeze(0).to(device)
    
    # Generate heatmap
    try:
        heatmap = gradcam.compute_heatmap(image_tensor, input_ids, attention_mask)
        overlay = gradcam.overlay_on_image(orig_image, heatmap, alpha=0.4)
        
        # Original
        axes[idx, 0].imshow(orig_image)
        axes[idx, 0].set_title('Original Image')
        axes[idx, 0].axis('off')
        
        # Heatmap
        axes[idx, 1].imshow(heatmap, cmap='jet', vmin=0, vmax=1)
        axes[idx, 1].set_title('Grad-CAM Heatmap')
        axes[idx, 1].axis('off')
        
        # Overlay
        axes[idx, 2].imshow(overlay)
        axes[idx, 2].set_title(f'Q: {sample["question"][:50]}...')
        axes[idx, 2].axis('off')
        
    except Exception as e:
        print(f'Error on sample {idx}: {e}')

plt.suptitle('Grad-CAM: Model Attention on Medical Images', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../experiments/gradcam_examples.png', dpi=150)
plt.show()

In [ ]:
# Error Analysis
print('=' * 50)
print('ERROR ANALYSIS')
print('=' * 50)

incorrect_indices = []
for i, (pred, ref) in enumerate(zip(results['predictions'], results['references'])):
    if pred.strip().lower().rstrip('.!?') != ref.strip().lower().rstrip('.!?'):
        incorrect_indices.append(i)

print(f'Total errors: {len(incorrect_indices)} / {len(results["predictions"])} ({100*len(incorrect_indices)/len(results["predictions"]):.1f}%)')
print(f'\nTop 10 errors (most confidently wrong):')
print()

if results['confidences']:
    # Sort by confidence (descending)
    incorrect_indices.sort(key=lambda i: results['confidences'][i], reverse=True)

for i, idx in enumerate(incorrect_indices[:10]):
    print(f'{i+1}. [{results["confidences"][idx]:.2f}] Q: {results["questions"][idx]}')
    print(f'   GT: "{results["references"][idx]}"  Pred: "{results["predictions"][idx]}"')
    print()

# Analysis of error types
yesno_errors = 0
open_errors = 0
for idx in incorrect_indices:
    if results['is_yesno'][idx]:
        yesno_errors += 1
    else:
        open_errors += 1

print(f'\nErrors by type:')
print(f'  Yes/No errors: {yesno_errors}')
print(f'  Open-ended errors: {open_errors}')

In [ ]:
# Results Summary Table
metrics = results['metrics']

print('=' * 60)
print('MEDVQA — FINAL RESULTS')
print('=' * 60)
print(f'{"Metric":<25} {"Score":<10}')
print('-' * 35)

key_metrics = ['yesno_accuracy', 'bleu_1', 'bleu_4', 'rouge_l_f1', 
               'bertscore_f1', 'ece', 'brier_score']

for m in key_metrics:
    if m in metrics:
        print(f'{m:<25} {metrics[m]:<10.4f}')

print('=' * 60)
print('\nResults also saved to experiments/ directory.')